In [5]:
import pandas as pd
import numpy as np

def calculate_clearing_price(my_price, my_vol, bids, asks):
    """
    Finds the clearing price that maximizes traded volume.
    bids/asks are dicts: {price: volume}
    """
    prices = sorted(list(set(list(bids.keys()) + list(asks.keys()) + [my_price])))
    
    best_price = 0
    max_traded_vol = -1
    
    for p in prices:
        # Sum of all bids >= p
        total_bids = sum(v for price, v in bids.items() if price >= p)
        if my_price >= p:
            total_bids += my_vol
            
        # Sum of all asks <= p
        total_asks = sum(v for price, v in asks.items() if price <= p)
        
        traded_vol = min(total_bids, total_asks)
        
        # Rule: Maximize volume, then choose higher price
        if traded_vol > max_traded_vol:
            max_traded_vol = traded_vol
            best_price = p
        elif traded_vol == max_traded_vol:
            if p > best_price:
                best_price = p
                
    return best_price, max_traded_vol

def optimize_strategy(bids, asks, buyback_price, fee=0):
    results = []
    
    # Try every price in the book
    potential_prices = sorted(list(set(list(bids.keys()) + list(asks.keys()))))
    
    # Try a range of volumes (Step through by 1000s or 500s)
    # Adjust range based on book depth
    for p_input in potential_prices:
        for v_input in range(0, 100000, 1000): 
            cp, total_v = calculate_clearing_price(p_input, v_input, bids, asks)
            
            # Remaining volume for you after others are filled
            my_fill = calculate_my_fill(p_input, v_input, cp, bids, asks)
            
            profit = my_fill * (buyback_price - cp - fee)
            results.append([p_input, v_input, cp, my_fill, profit])
            
    df = pd.DataFrame(results, columns=['MyBidPrice', 'MyBidVol', 'ClearingPrice', 'MyFill', 'Profit'])
    return df.sort_values(by='Profit', ascending=False).head(1)

# --- DATA INPUT AREA ---
# Fill in the actual prices from your UI screenshot!
flax_bids = {30: 30000, 29: 5000, 28: 12000, 27: 28000}
flax_asks = {28: 40000, 31: 20000, 32: 20000, 33: 30000}

mushroom_bids = {20: 43000, 19: 17000, 18: 6000, 17: 5000, 16: 10000, 15: 5000, 14: 10000, 13: 7000} 
mushroom_asks = {12: 20000, 13: 25000, 14: 35000, 15: 6000, 16: 5000, 18: 10000, 19: 12000} 

print("Best Flax Strategy:")
print(optimize_strategy(flax_bids, flax_asks, 30))

print("\nBest Mushroom Strategy:")
print(optimize_strategy(mushroom_bids, mushroom_asks, 20, fee=0.10))

Best Flax Strategy:
     MyBidPrice  MyBidVol  ClearingPrice  MyFill  Profit
509          32      9000             29    9000    9000

Best Mushroom Strategy:
     MyBidPrice  MyBidVol  ClearingPrice  MyFill   Profit
740          19     40000             18   40000  76000.0


In [4]:
def calculate_my_fill(my_price, my_vol, cp, bids, asks):
    # Total supply available at the clearing price or cheaper
    total_supply = sum(v for p, v in asks.items() if p <= cp)
    
    # Volume of people who have priority over you:
    # 1. Everyone bidding a higher price than you
    # 2. Everyone bidding the same price as you (because you are last)
    volume_ahead_of_me = sum(v for p, v in bids.items() if p >= my_price)
    
    # Remaining supply for you
    fill = max(0, total_supply - volume_ahead_of_me)
    
    # You can't buy more than you asked for
    return min(my_vol, fill)